# 07 · Profiling

This notebook measures where decode time goes:
- each compiled entry point across prefix lengths;
- the per-kernel breakdown from TVM's VM profiler;
- what the 28 attention callbacks cost;
- a full refinement step, split into parts, next to the same step in HF.

**Needs:** CUDA, `artifacts/fast_dllm_relax_bf16`, and the checkpoint (last section). A few minutes.
Absolute numbers depend on the GPU; `env_report()` in notebook 00 records which one ran.

In [1]:
from _helpers import *
bootstrap()

repo root: fast-dllm-relax/  (cwd set)  tvm 0.24.dev0  torch 2.10.0+cu128


PosixPath('<repo>')

In [2]:
import collections
import statistics
import time
import numpy as np
import torch
import tvm
from fast_dllm_relax import runtime as rt
from fast_dllm_relax.model.cuda import build_prefix_block_attention_bias

runner = load_relax("artifacts/fast_dllm_relax_bf16")
vm, P, dev, reshape = runner.vm, runner.params, runner.dev, runner.reshape
NL, NKV, HD, H = 28, 2, 128, 1536
rng = np.random.default_rng(0)

def embed(n):
    t = tvm.runtime.tensor(rng.integers(0, 150000, n).astype("int32"), device=dev)
    return reshape(vm["embed"](t, P), tvm.runtime.ShapeTuple([1, n, H]))

def cache(n):
    t = (0.5 * torch.randn(NL, 1, NKV, n, HD, device="cuda")).to(torch.bfloat16)
    return tvm.runtime.from_dlpack(t)

def offset(n):
    return tvm.runtime.tensor(np.asarray(n, "int32"), device=dev)

def bench(f, *args, n=30, warmup=5):
    for _ in range(warmup):
        f(*args)
    dev.sync()
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); f(*args); dev.sync(); ts.append(time.perf_counter() - t0)
    return statistics.median(ts) * 1e3

## Time per entry point

In [3]:
PREFIXES = [64, 256, 704, 2048]
rows = []
e32, e8 = embed(32), embed(8)
bk, bv = cache(32), cache(32)
for p in PREFIXES:
    k, v, off = cache(p), cache(p), offset(p)
    bias = tvm.runtime.tensor(build_prefix_block_attention_bias(0, p, 12, block_size=32), device=dev)
    ep = embed(p)
    rows.append({
        "prefix": p,
        "build_prefix_cache": bench(vm["build_prefix_cache"], ep, bias, P, n=10),
        "forward_cached (append)": bench(vm["forward_cached"], e32, k, v, off, P),
        "forward_cached_block (refine)": bench(vm["forward_cached_block"], e32, k, v, off, P),
        "forward_block_partial_rp8": bench(vm["forward_block_partial_rp8"], e8, k, v, bk, bv, offset(p + 8), P),
    })
hid32 = vm["forward_cached_block"](e32, cache(64), cache(64), offset(64), P)[0]
hid8 = vm["forward_block_partial_rp8"](e8, cache(64), cache(64), bk, bv, offset(72), P)[0]
print_table(rows, floatfmt=".3f")
print(f"get_logits: 32 rows {bench(vm['get_logits'], hid32, P):.3f} ms   8 rows {bench(vm['get_logits'], hid8, P):.3f} ms")

prefix  build_prefix_cache  forward_cached (append)  forward_cached_block (refine)  forward_block_partial_rp8
------  ------------------  -----------------------  -----------------------------  -------------------------
64      9.465               8.691                    8.664                          8.318                    
256     14.163              9.046                    8.923                          8.532                    
704     35.826              9.629                    9.323                          8.899                    
2048    175.676             11.058                   10.228                         9.817                    
get_logits: 32 rows 1.315 ms   8 rows 1.219 ms


- `forward_cached_block` is the refinement step and runs several times per block.
- `forward_cached` runs once per block. The gap between them is the restack of the full cache that
  `forward_cached_block` skips.
- `build_prefix_cache` runs once per prompt. In bf16 its attention matmuls are dlight-scheduled
  TIR (notebook 04 (e)), so its cost grows quadratically with the prefix.

## The VM profiler

`VirtualMachine(ex, dev, profile=True).profile(fn, *args)` times every instruction: TIR kernels,
cuBLAS calls and the Python callback.

In [4]:
pvm = tvm.relax.VirtualMachine(runner.loaded_ex, dev, profile=True)
p = 704
report = pvm.profile("forward_cached_block", e32, cache(p), cache(p), offset(p), P)
lines = str(report.table(sort=True, aggregate=True, col_sums=True)).splitlines()
print("\n".join(lines[:30]))
if len(lines) > 30:
    print(f"... ({len(lines) - 30} more lines)")

Name                                                                                                                                                                                                                                                                                                                                                                                            Duration (us)  Percent  Device  Count                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

[17:28:10] <repo>/3rdparty/tvm/src/runtime/threading_backend.cc:348: Warning: more than two frequencies detected! Forced big_count_ to 16
[17:28:10] <repo>/3rdparty/tvm/src/runtime/threading_backend.cc:348: Warning: more than two frequencies detected! Forced big_count_ to 16


## What the attention callback costs

Swap the callback for a stub that returns a preallocated tensor of the right shape (wrong values, no
work), build a VM against the stub, and time the same calls. The difference is the cost of the 28
callbacks: SDPA itself plus the trip from VM to Python and back. A separate measurement times SDPA
alone at the same shapes.

In [5]:
orig_cb = tvm.get_global_func("fast_dllm.flash_attn_bf16")
_out = {}
def stub_cb(q, k, v):
    key = tuple(q.shape)
    if key not in _out:
        _out[key] = tvm.runtime.from_dlpack(torch.zeros(key, dtype=torch.bfloat16, device="cuda"))
    return _out[key]
tvm.register_global_func("fast_dllm.flash_attn_bf16", stub_cb, override=True)
vm_stub = tvm.relax.VirtualMachine(runner.loaded_ex, dev)
tvm.register_global_func("fast_dllm.flash_attn_bf16", orig_cb, override=True)

rows = []
for p in PREFIXES:
    k, v, off = cache(p), cache(p), offset(p)
    real = bench(vm["forward_cached_block"], e32, k, v, off, P)
    stub = bench(vm_stub["forward_cached_block"], e32, k, v, off, P)
    q = torch.randn(1, 12, 32, HD, device="cuda", dtype=torch.bfloat16)
    kk = torch.randn(1, NKV, p + 32, HD, device="cuda", dtype=torch.bfloat16)
    def sdpa():
        torch.nn.functional.scaled_dot_product_attention(q, kk, kk, scale=HD ** -0.5, enable_gqa=True)
    for _ in range(5): sdpa()
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(30): sdpa()
    torch.cuda.synchronize(); sd = (time.perf_counter() - t0) / 30 * 1e3
    rows.append({"prefix": p, "step ms": real, "with stub ms": stub, "28 callbacks ms": real - stub,
                 "share of step": (real - stub) / real, "SDPA alone x28 ms": 28 * sd})
print_table(rows, floatfmt=".3f")

prefix  step ms  with stub ms  28 callbacks ms  share of step  SDPA alone x28 ms
------  -------  ------------  ---------------  -------------  -----------------
64      8.689    8.470         0.219            0.025          0.208            
256     8.938    8.587         0.350            0.039          0.267            
704     9.321    8.830         0.491            0.053          0.365            
2048    10.225   9.573         0.652            0.064          0.498            


`28 callbacks` minus `SDPA alone x28` is roughly the round-trip overhead: the VM → Python call, DLPack
wrapping, the transposes, and synchronization at the TVM/PyTorch boundary. That overhead is what an
in-graph attention kernel would remove (research 001, open question 4). Metal's build already
computes attention in-graph (notebook 08).

## One refinement step, split up, next to HF

In [6]:
def sync_time(f):
    dev.sync(); torch.cuda.synchronize(); t0 = time.perf_counter(); r = f(); dev.sync(); torch.cuda.synchronize()
    return r, (time.perf_counter() - t0) * 1e3

p = 704
k, v, off = cache(p), cache(p), offset(p)
ids32 = tvm.runtime.tensor(rng.integers(0, 150000, 32).astype("int32"), device=dev)
parts = collections.defaultdict(list)
for _ in range(30):
    e, t = sync_time(lambda: reshape(vm["embed"](ids32, P), tvm.runtime.ShapeTuple([1, 32, H]))); parts["embed"].append(t)
    out, t = sync_time(lambda: vm["forward_cached_block"](e, k, v, off, P)); parts["forward_cached_block"].append(t)
    lg, t = sync_time(lambda: vm["get_logits"](out[0], P)); parts["get_logits"].append(t)
    def reduce():
        l = torch.from_dlpack(lg); s = torch.cat([l[:, :1], l[:, :-1]], 1)
        return s.argmax(-1)[0].cpu().numpy(), s.softmax(-1).max(-1).values[0].cpu().numpy()
    _, t = sync_time(reduce); parts["torch shift/softmax/argmax + .cpu()"].append(t)
rows = [{"part": k_, "ms": statistics.median(v_)} for k_, v_ in parts.items()]
rows.append({"part": "total", "ms": sum(r["ms"] for r in rows)})
print_table(rows, floatfmt=".3f")

part                                 ms    
-----------------------------------  ------
embed                                0.038 
forward_cached_block                 9.368 
get_logits                           1.318 
torch shift/softmax/argmax + .cpu()  0.227 
total                                10.951


In [7]:
del runner, vm, pvm, vm_stub; torch.cuda.empty_cache()
tok, hf = load_hf()
with torch.no_grad():
    ctx = torch.randint(0, 150000, (1, p), device="cuda")
    past = hf(input_ids=ctx, use_cache=True, update_past_key_values=True).past_key_values
    blk = torch.randint(0, 150000, (1, 32), device="cuda")
    def hf_step():
        out = hf(input_ids=blk, use_cache=True, past_key_values=past, update_past_key_values=False)
        s = torch.cat([out.logits[:, :1], out.logits[:, :-1]], 1)
        return torch.softmax(s, -1).max(-1)
    for _ in range(5): hf_step()
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(30): hf_step()
    torch.cuda.synchronize()
print(f"HF eager refinement step at prefix {p}: {(time.perf_counter() - t0) / 30 * 1e3:.3f} ms")

HF eager refinement step at prefix 704: 13.140 ms


## Takeaways
- The refinement step (`forward_cached_block` + `get_logits` + reductions) dominates decode time.
  Prefill (`build_prefix_cache`) is paid once per prompt.
- The VM profiler shows which kernels matter. cuBLAS GEMMs should lead; any TIR kernel near the top
  is a scheduling opportunity.
- The callback's round trip is a measurable, fixed per-layer cost. It is the main argument for
  in-graph or FlashInfer attention on CUDA.
- Next: **08 · Other targets**.